In [ ]:

import numpy as np

from sklearn.datasets import load_breast_cancer, make_moons
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline


# 1. Build SVC Pipeline
def build_svc_pipeline(C=1.0, gamma="scale", kernel="rbf"):

    pipeline = Pipeline([
        ("scaler", StandardScaler()),
        ("svc", SVC(
            C=C,
            gamma=gamma,
            kernel=kernel
        ))
    ])

    return pipeline


# 2. Scaling Effect
def scaling_effect(random_state=0):

    d = load_breast_cancer()

    X = d.data
    y = d.target

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.20,
        stratify=y,
        random_state=random_state
    )

    # Unscaled SVC
    raw_model = SVC(kernel="rbf")

    raw_model.fit(X_train, y_train)

    unscaled_acc = raw_model.score(X_test, y_test)

    # Scaled SVC
    scaled_model = build_svc_pipeline()

    scaled_model.fit(X_train, y_train)

    scaled_acc = scaled_model.score(X_test, y_test)

    return {
        "unscaled_acc": round(float(unscaled_acc), 3),
        "scaled_acc": round(float(scaled_acc), 3)
    }


# 3. Kernel Effect
def kernel_effect(noise=0.20, random_state=0):

    X, y = make_moons(
        n_samples=600,
        noise=noise,
        random_state=random_state
    )

    cv = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=random_state
    )

    # Linear Kernel
    linear_model = build_svc_pipeline(
        kernel="linear"
    )

    linear_scores = cross_val_score(
        linear_model,
        X,
        y,
        cv=cv,
        scoring="accuracy"
    )

    # RBF Kernel
    rbf_model = build_svc_pipeline(
        kernel="rbf"
    )

    rbf_scores = cross_val_score(
        rbf_model,
        X,
        y,
        cv=cv,
        scoring="accuracy"
    )

    return {
        "linear_acc": round(float(np.mean(linear_scores)), 3),
        "rbf_acc": round(float(np.mean(rbf_scores)), 3)
    }


# 4. Number of Support Vectors
def n_support_vectors(C=1.0, random_state=0):

    X, y = make_moons(
        n_samples=600,
        noise=0.20,
        random_state=random_state
    )

    model = build_svc_pipeline(
        C=C,
        kernel="rbf"
    )

    model.fit(X, y)

    svc = model.named_steps["svc"]

    return int(svc.support_vectors_.shape[0])


# Verification
print("Scaling Effect:")
print(scaling_effect(0))


print("\nKernel Effect:")
print(kernel_effect(0.20, 0))


print("\nSupport Vectors:")
print(n_support_vectors(0.1))
print(n_support_vectors(10.0))


# Additional Pipeline Verification
print("\nPipeline:")
print(build_svc_pipeline())

print("\nCustom Pipeline:")
print(build_svc_pipeline(C=10, gamma=0.1, kernel="linear"))
